# Nemotron AI-Q: Building a Personal Assistant

In this notebook, we will use **AI-Q** to build a a domain-specific **personal assistant** agent. 

**NVIDIA Nemotron** will be leveraged as LLM model, which is a family of open models with open weights, training data, and recipes, delivering leading efficiency and accuracy for building specialized AI agents. We will first learn how to deploy Nemotron model from Google Cloud **Vertex AI Model Garden**.

Then we will use a **blueprint** — a pre-built, production-ready agent workflow that you can run as-is or customize for your domain. 

**AI-Q** is NVIDIA's deep research agent blueprint, built on top of NeMo Agent Toolkit (NAT). Give it a question, and it will:

1. **Clarify** your intent by asking follow-up questions
2. **Plan** a research strategy with search queries
3. **Research** the topic across multiple web searches
4. **Produce** a comprehensive, cited report

**AI-Q** is designed to be customized through three levers:

1. **Prompts** — Tell the agent what domain it specializes in
2. **Tools** — Give the agent access to domain-specific data
3. **Configuration** — Wire the tools into the workflow

By the end of this notebook, our personal assistant will answer any question you have about NVIDIA and Google collaborations, cite relevant information (e.g. blog, code base) with its advanced web search capabilities.

## What You'll Learn

- What **Nemotron** model is
- Deploy **Nemotron** model from Vertex AI Model Garden
- What a **blueprint** is and how AI-Q works
- **Customize** agent behavior with domain specific prompt and tool
- **Run** the personal assistant agent and interpret its output
- **Deploy** AI-Q as a **web application** with Docker Compose


#### Install pacakges

In [ ]:
! pip3 install --upgrade --quiet google-cloud-aiplatform
! pip3 install -U -q httpx

In [ ]:
import json
import time

In [ ]:
import os
os.environ["PATH"] = "/home/jupyter/.local/bin:" + os.environ["PATH"]
os.environ["PATH"] = ".venv/bin:" + os.environ["PATH"]

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
!source ~/.bashrc
!uv python install 3.12
!uv venv --python 3.12 --clear
!uv pip install -e aiq -q
!uv pip install -e aiq/sources/tavily_web_search -q
!uv pip install -e aiq/sources/google_scholar_paper_search -q
!uv pip uninstall chardet
!uv pip install --upgrade "requests==2.32.3" "urllib3==2.3.0" "charset-normalizer==3.4.1"
# https://github.com/NVIDIA-AI-Blueprints/aiq/blob/main/pyproject.toml

In [ ]:
!uv pip list | grep -E "requests|urllib3|charset-normalizer|chardet"

#### Define environment variables

Open `.env` in the Terminal and fill in your keys:

| Key | Where to get it |
|-----|-----------------|
| `TAVILY_API_KEY` | Sign up at [app.tavily.com](https://app.tavily.com) → API Keys (free tier available) |

```
TAVILY_API_KEY=tvly-xxxxxxxxxxxxxxxxxxxx
```

> **Keep this file private.** It's already listed in `.gitignore` so it won't be committed, but don't share it.


In [ ]:
from dotenv import find_dotenv, load_dotenv
# Make sure .env file is in the same directory as the notebook
load_dotenv(find_dotenv())

assert os.getenv("TAVILY_API_KEY"), "TAVILY_API_KEY not found — check your .env file"
print("API keys loaded successfully.")

## 1. What is Nemotron

NVIDIA Nemotron™ is a family of open models with open weights, training data, and recipes, delivering leading efficiency and accuracy for building specialized AI agents.

### Available `Nemotron` models on Vertex AI

#### `Nemotron-3-Super-120B-A12B`
**Nemotron-3-Super-120B-A12B** is a large language model (LLM) trained by NVIDIA, designed to deliver strong agentic, reasoning, and conversational capabilities. It is optimized for collaborative agents and high-volume workloads such as IT ticket automation. Like other models in the family, it responds to user queries and tasks by first generating a reasoning trace and then concluding with a final response. The model's reasoning capabilities can be configured through a flag in the chat template.

#### `Llama-3.3-Nemotron-Super-49B-v1.5`
**Llama-3.3-Nemotron-Super-49B-v1.5** is a reasoning model that is post trained for reasoning, human chat preferences, and agentic tasks, such as Retrieval-Augmented Generation (RAG) and tool calling. The model supports a context length of 128K tokens. Llama-3.3-Nemotron-Super-49B-v1.5 is a model which offers a great tradeoff between model accuracy and efficiency. Efficiency (throughput) directly translates to savings. Using a novel Neural Architecture Search (NAS) approach, we greatly reduce the model’s memory footprint, enabling larger workloads, as well as fitting the model on a single GPU at high workloads (H200). This NAS approach enables the selection of a desired point in the accuracy-efficiency tradeoff. For more information on the NAS approach, please refer to this [paper](https://arxiv.org/abs/2411.19146).

#### `Nemotron Nano 12B V2 VL`
**NVIDIA Nemotron Nano 12B v2 VL** model enables multi-image reasoning and video understanding, along with strong document intelligence, visual Q&A and summarization capabilities.

Nemotron Nano 12B V2 VL is a model for multi-modal document intelligence. It would be used by individuals or businesses that need to process documents such as invoices, receipts, and manuals. The model is capable of handling multiple images of documents, up to four images at a resolution of 1k x 2k each, along with a long text prompt. The expected use is for tasks like summarization and Visual Question Answering (VQA). The model is also expected to have a significant advantage in throughput.

### Reference(s)

* [NVIDIA Nemotron 3 model family on Hugging Face](https://huggingface.co/collections/nvidia/nvidia-nemotron-v3)
* [NVIDIA Nemotron 2 model family on Hugging Face](https://huggingface.co/collections/nvidia/nvidia-nemotron-v2)
* [NVIDIA Nemotron 3 White Paper](https://arxiv.org/abs/2512.20856)

## 2. Deploy Nemotron from Vertex AI Model Garden

### Setup Variables

In [ ]:
# Please set below
PUBLISHER_NAME = "nvidia"
PUBLISHER_MODEL_NAME = "llama-nemotron-super" 
LOCATION = "us-central1" 

### Set Google Cloud project information and initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com). Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [ ]:
# Please set below
PROJECT_ID = [your_project_id]
ENDPOINT = f"https://{LOCATION}-aiplatform.googleapis.com"

In [ ]:
from google.cloud import aiplatform
aiplatform.init(project=PROJECT_ID, location=LOCATION)

### Upload Model

In [ ]:
MODEL_NAME="ModelGarden_LaunchPad_Endpoint_" + time.strftime("%Y%m%d-%H%M%S")
model = aiplatform.Model.upload(
    display_name=MODEL_NAME,
    model_garden_source_model_name = f"publishers/{PUBLISHER_NAME}/models/{PUBLISHER_MODEL_NAME}",
)

### Create Endpoint

In [ ]:
my_endpoint = aiplatform.Endpoint.create(display_name="ModelGarden_LaunchPad_Endpoint_" + time.strftime("%Y%m%d-%H%M%S"))

### Deploy Model

Deploy the model to the endpoint. The deployment process will take a few minutes.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

# Configure machine types (please change if needed)

MACHINE_CONFIG = {
    "a2-highgpu-8g":  {"type": "NVIDIA_TESLA_A100",       "count": 8},
    "a2-ultragpu-8g": {"type": "NVIDIA_A100_80GB",        "count": 8},
    "a3-highgpu-4g":  {"type": "NVIDIA_H100_80GB",        "count": 4},
    "g4-standard-384": {"type": "NVIDIA_RTX_PRO_6000",    "count": 8},
}

COMPATIBLE_MACHINES = list(MACHINE_CONFIG.keys())

# Setup Widgets

dropdown_machine = widgets.Dropdown(
    options=COMPATIBLE_MACHINES,
    description="Machine type:",
    style={"description_width": "initial"},
)

label_accelerator_type = widgets.HTML()
label_accelerator_count = widgets.HTML()


def update_labels(machine_type: str):
    """Update accelerator labels based on the selected machine."""
    info = MACHINE_CONFIG.get(machine_type, {"type": "Unknown", "count": 0})
    label_accelerator_type.value = f"<b>Accelerator type:</b> {info['type']}"
    label_accelerator_count.value = f"<b>Accelerator count:</b> {info['count']}"
    return info


# Event Handling

def on_machine_change(change):
    """Handle machine selection change."""
    if change["name"] == "value" and change["type"] == "change":
        global MACHINE_TYPE, ACCELERATOR_TYPE, ACCELERATOR_COUNT
        MACHINE_TYPE = change.new
        info = update_labels(MACHINE_TYPE)
        ACCELERATOR_TYPE = info["type"]
        ACCELERATOR_COUNT = info["count"]


# Initiate Global Variables

MACHINE_TYPE = dropdown_machine.value
initial_info = update_labels(MACHINE_TYPE)
ACCELERATOR_TYPE = initial_info["type"]
ACCELERATOR_COUNT = initial_info["count"]

dropdown_machine.observe(on_machine_change, names="value")

display(
    widgets.VBox([
        dropdown_machine,
        label_accelerator_type,
        label_accelerator_count,
    ])
)

In [ ]:
model.deploy(
    endpoint=my_endpoint,
    deployed_model_display_name=MODEL_NAME,
    traffic_split={"0": 100},
    machine_type=MACHINE_TYPE,
    accelerator_type=ACCELERATOR_TYPE,
    accelerator_count=ACCELERATOR_COUNT,
    min_replica_count=1,
    max_replica_count=1,
    spot=True  # if quota only available in `VertexAI API Custom model serving preemptible`
)

### Prediction

Sends a POST request to the specified API endpoint to get a response from the model for a limerick writing using the provided payload, with reasoning on / off.

In [ ]:
# Reasoning on
PAYLOAD_MODEL_NAME = "llama-3.3-nemotron-super-49b-v1.5"
PAYLOAD = {
    "model": PUBLISHER_NAME + "/" + PAYLOAD_MODEL_NAME,
    "messages": [
       {"role": "system", "content": "/think"}, 
       {"role":"user", "content":"How many 'r's are in 'strawberry'?"}
    ],
    "temperature": 0.6,
    "top_p": 0.95,
    "max_tokens": 1024,
    "stream": False
}

request = json.dumps(PAYLOAD)

response = my_endpoint.raw_predict(
    body = request,
    headers = {'Content-Type':'application/json'}
)

result = json.loads(response.text)
print(json.dumps(result["choices"][0]["message"]["content"]))

## 3. What is AI-Q?

AI-Q is a **deep research agent** — an AI system that goes far beyond a single LLM call. When you ask it a question, it doesn't just generate an answer from memory. Instead, it runs a multi-stage pipeline:

| Stage | What Happens |
|-------|-------------|
| **Intent Classification** | Determines what kind of research is needed (web search, academic papers, or both) |
| **Clarification** | Asks you follow-up questions to refine the research scope (human-in-the-loop) |
| **Planning** | Builds a structured research plan: table of contents, search queries, acceptance criteria |
| **Deep Research** | Executes searches across multiple rounds, synthesizing findings as it goes |
| **Report Generation** | Produces a publication-ready report with numbered citations `[1][2]` |

Under the hood, AI-Q is a NAT workflow — just like the ones you built in Module 2, but significantly more sophisticated. It composes multiple specialized agents (intent classifier, clarifier, planner, researcher) into a single orchestrated pipeline.

### What's a Blueprint?

A **blueprint** is a pre-built, production-ready agent workflow. Think of it as a complete application template:

- **Pre-built**: All the agent logic, prompts, and tools are already written
- **Production-ready**: Designed for real-world use with error handling, retries, and checkpointing
- **Customizable**: You can swap out models, add tools, change prompts, and adjust configs

You don't need to build from scratch — you configure and extend what's already there.

### The Three Customization Levers

Let's understand the three ways to customize AI-Q:

| Lever | What It Controls | File Type |
|-------|-----------------|----------|
| **Prompts** | What the agent *knows* about your domain — its persona, priorities, and citation style | Jinja2 template (`.j2`) |
| **Tools** | What data the agent can *access* — local databases, APIs, internal knowledge | Python package registered with NAT |
| **Config** | How tools and agents are *wired together* — which tools the researcher uses, in what order | YAML config (`.yml`) |

### Architecture Overview

The diagram below shows how AI-Q's components connect. AI-Q is built, orchestrated, evaluated, and deployed using NVIDIA's [**NeMo Agent Toolkit**](https://developer.nvidia.com/nemo-agent-toolkit). The top-level workflow is a [**LangGraph**](https://langchain-ai.github.io/langgraph/) state machine that routes every query through an Intent Classifier, then down one of two paths: a fast **Shallow Research Agent** for simple lookups, or a full **Deep Research Agent** for complex, multi-faceted topics.

The deep path is where AI-Q really shines — it uses a hierarchical multi-agent system built with [**LangChain**](https://www.langchain.com/) where an **Orchestrator** delegates work to specialized sub-agents (Planner and Researcher), coordinates their output through a virtual filesystem, and synthesizes everything into a publication-ready report.

![AI-Q Architecture](./images/aiq-architecture.png)

**How the pieces fit together:**

- The top-level workflow is a **LangGraph state machine** that manages routing, state, and checkpointing. Each agent node (Intent Classifier, Shallow Research, Clarifier) is a LangGraph `StateGraph` with conditional edges that determine the next step.
- The **Intent Classifier** makes a single-pass decision: is this a casual question (meta), a quick lookup (shallow), or a complex research topic (deep)?
- The **Shallow Research Agent** is a LangGraph agent loop that handles simple queries with bounded tool calls (max ~5). If it can't produce a confident answer, it **escalates** to the deep path.
- The **Clarifier Agent** (optional, disabled in our training config) is another LangGraph graph that can ask 1-2 follow-up questions and generate a research plan for user approval before deep research begins.
- The **Deep Research Agent** uses LangChain's `deepagents` library for hierarchical multi-agent orchestration. The Orchestrator delegates to sub-agents: first the Planner creates a structured research plan (TOC, search queries, acceptance criteria), then the Researcher executes those queries across multiple rounds. Sub-agents communicate through a shared virtual filesystem (`/shared/`), and the Orchestrator synthesizes everything into a final report (3,000-5,000+ words with numbered citations).
- All **research tools** (web search, paper search, knowledge search) are implemented as LangChain tools (`BaseTool`), making them composable across any agent in the pipeline.

## 4. Customize agent behavior with domain specific prompt and tool

### Customize prompt

The orchestrator agent's behavior is guided by a **system prompt** — a Jinja2 template file. This is just a text file that the agent reads on every run. By adding domain-specific context at the top, we can steer the agent toward tax-focused research.

Let's first look at the current prompt:

In [ ]:
from IPython.display import display, Markdown

prompt_path = "aiq/src/aiq_agent/agents/deep_researcher/prompts/orchestrator.j2"

with open(prompt_path) as f:
    original_prompt = f.read()

# Show the first 20 lines
for i, line in enumerate(original_prompt.splitlines()[:20], 1):
    print(f"{i:3d} | {line}")

print(f"\n... ({len(original_prompt.splitlines())} total lines)")

The first line says:

> *"You are a Deep Research agent that coordinates the production of a publication-ready, well-cited research report on the provided topic."*

That's generic. Let's add a **domain context block** at the very top that tells the agent it specializes in a specific domain. We don't need to change any Python code — we just prepend text to this file.

In [ ]:
domain_header = """## Domain Context: NVIDIA Google Cloud Assistant
This agent specializes in the collaborations between NVIDIA and Google Cloud. 
It could provide information about software integrations (e.g. NIM, TensorRT-LLM, Dynamo) and hardware accelerations (e.g. B200, GB200, RTX PRO 6000). 
Always cite the related launch blogs, code bases, or example guides to enable user to learn the business and technical details.
"""

# Prepend the domain header to the original prompt
updated_prompt = domain_header + original_prompt

with open(prompt_path, "w") as f:
    f.write(updated_prompt)

print("Orchestrator prompt updated with domain context.")
print("\nFirst 10 lines of the updated prompt:")
print("─" * 60)
for line in updated_prompt.splitlines()[:10]:
    print(line)

### Customize Configurations

The configuration has three main sections:

**`llms`** — Three instances of the same model (Nemotron), each tuned for a different job:

| LLM | Temperature | Purpose |
|-----|-------------|--------|
| `nemotron_llm_intent` | 0.5 | Intent classification — needs some creativity to interpret queries |
| `nemotron_llm` | 0.1 | General reasoning — needs precision for planning and clarification |
| `nemotron_llm_deep` | 1.0 | Deep synthesis — needs maximum creativity for comprehensive reports |

All three use the `NVIDIA Nemotron` model via Vertex AI Endpoint, from which we just deployed from Model Garden.

**`functions`** — The tools and agent components:
- `web_search_tool` / `advanced_web_search_tool` — Tavily web search (your Tavily API key)
- `intent_classifier` → `clarifier_agent` → `deep_research_agent` — The agent pipeline

**`workflow`** — The orchestrator:
- `_type: chat_deepresearcher_agent` — The top-level workflow that chains everything together
- `enable_clarifier: false` — Disabled for this training (when enabled, asks follow-up questions before researching)
- `enable_escalation: true` — Can escalate from shallow to deep research

To customize the configuration, we could add new tool available in NAT to introduce more functionality. Details could refer to [here](https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/functions-and-function-groups/functions.html#agents-and-tools).
We could also create customized new tool that reads data from local data source, then wires the tool into both agents. 

The current configurations use Nemotron model for intent classifier, shallow research agent and deep research agent, and enables web research tool with Tavily.

In [ ]:
with open("configs/config_cli_vertex.yml") as f:
    print(f.read())

## 5. Running the Personal Assistant Agent

Let's run AI-Q with a sample query. We use `nat run` — pointing it at AI-Q's configuration file.

Since this is a deep research agent, it performs multiple rounds of search and synthesis. **This may take a couple of minutes to complete.** Output will appear when it finishes.

The agent will be specific to the domain we set up earlier, generate a more focused and detailed report structure.

> **Note:** If you see an SSL error, timeout, or connection error, it is likely a transient network issue with the inference endpoint — simply re-run the cell.

In [ ]:
import os
import subprocess

# Please set below
## Vertex endpoint
os.environ["PROJECT"] = [your_project]
os.environ["REGION"] = [your_region]
os.environ["ENDPOINT_ID"] = [your_endpoint_id]
os.environ["MODEL"] = PUBLISHER_NAME + "/" + PAYLOAD_MODEL_NAME

### Run Proxy

*Note: Vertex custom endpoints (NIM) only support `:rawPredict`, not `/chat/completions`. The shallow/deep researcher use an OpenAI client that calls base_url + `/chat/completions`, which returns 400 "Precondition check failed" on Vertex. So we point ENDPOINT_URL at a local proxy that forwards `/chat/completions"` to Vertex `:rawPredict`. Start the proxy first (see below).*

In [ ]:
# Proxy: run in a separate terminal (or background) before running the workflow:
#   python vertex_chat_completions_proxy.py 8765
# Then set ENDPOINT_URL to the proxy so the OpenAI client hits the proxy, which forwards to Vertex.
PROXY_PORT = 8765
os.environ["ENDPOINT_URL"] = f"http://localhost:{PROXY_PORT}/v1"

# Same gcloud token for OPENAI_API_KEY (proxy forwards it to Vertex)
token = subprocess.check_output(
    ["gcloud", "auth", "print-access-token"],
    text=True,
).strip()
os.environ["OPENAI_API_KEY"] = token

# Verify
print("ENDPOINT_URL (proxy):", os.environ["ENDPOINT_URL"])
print("OPENAI_API_KEY: set" if os.environ.get("OPENAI_API_KEY") else "OPENAI_API_KEY: not set")
print("Make sure vertex_chat_completions_proxy.py is running on port", PROXY_PORT)

In the Terminal, first define the endpoint variables

`export REGION=[your_region] PROJECT=[your_project_id] ENDPOINT_ID=[your_endpoint_id]`

Then run the proxy and keep the connection alive. 

`python vertex_chat_completions_proxy.py 8765` 

### Run the Agent

In [ ]:
!nat run --config_file configs/config_cli_vertex.yml \
    --input "What are collaborations between NVIDIA and Google Cloud for accelerated computing?"

## 6. Deploy with Docker Compose (Web UI)

So far we've run AI-Q from the **command line**. For a shareable, multi-user experience, we can deploy the full stack as a **web application** using Docker Compose — complete with a browser-based chat UI, async job management, and knowledge upload.

### Architecture

The Docker Compose stack runs four services:

| Service | Image | Port | Role |
|---------|-------|------|------|
| `vertex-proxy` | `python:3.12-slim` | 8765 (internal) | Translates `/v1/chat/completions` → Vertex `:rawPredict` |
| `aiq-agent` | Built from `deploy/Dockerfile` | 8000 | Backend API server (FastAPI) |
| `aiq-blueprint-ui` | Built from `frontends/ui` | 3000 | Next.js chat frontend |
| `postgres` | `postgres:16-alpine` | 5432 | Async jobs + LangGraph checkpoints |

```
Browser → aiq-blueprint-ui:3000 → aiq-agent:8000 → vertex-proxy:8765 → Vertex AI :rawPredict
```

### Key Differences from CLI Mode

**1. Web config** (`config_web_vertex.yml`) adds a `front_end` section that launches the FastAPI server with CORS, async jobs, and knowledge API. All LLMs use `_type: openai` pointing at the proxy via the `ENDPOINT_URL` environment variable.

**2. Containerized proxy with auto-token** — Inside Docker on GCE, the proxy fetches access tokens from the **GCE metadata server** automatically. No need for `gcloud auth print-access-token` or manual token refresh. `OPENAI_API_KEY` is set to `placeholder`.

In [ ]:
with open("configs/config_web_vertex.yml") as f:
    print(f.read())

### Configure Environment

The Docker Compose stack reads variables from `deploy/.env`:

In [ ]:
!cp deploy/.env.example deploy/.env

Please edit below variables in `deploy/.env`
```
BACKEND_CONFIG=/app/configs/config_web_vertex.yml
TAVILY_API_KEY={os.environ.get('TAVILY_API_KEY', 'YOUR_TAVILY_KEY')}
PROJECT={os.environ.get('PROJECT', PROJECT_ID)}
REGION={LOCATION}
ENDPOINT_ID={os.environ.get('ENDPOINT_ID', 'YOUR_ENDPOINT_ID')}
MODEL=nvidia/llama-3.3-nemotron-super-49b-v1.5
ENDPOINT_URL=http://vertex-proxy:8765/v1
OPENAI_API_KEY=placeholder
```

In [ ]:
env_path = "deploy/.env"
with open(env_path) as f:
    env_content = f.read()

print(env_content)

### Docker Compose

Docker Compose brings up all three services (backend, frontend, database) with a single command.

In [ ]:
!cd deploy/compose && docker compose --env-file ../.env -f docker-compose.yaml up -d --build

In [ ]:
import subprocess                                                                                                                                                                                                                                  
print(subprocess.check_output(                                                                                                                                                                                                                 
      ["docker", "ps", "--filter", "name=aiq",                                                                                                                                                                                                       
       "--format", "table {{.Names}}\t{{.Status}}\t{{.Ports}}"],                                                                                                                                                                                     
      text=True                                                                                                                                                                                                                                      
  ))

## or in Terminal
# docker ps --filter "name=aiq" --format "table {{.Names}}\t{{.Status}}\t{{.Ports}}"

### Accessing the UI

On a remote GCE instance, set up an **SSH tunnel** to forward port 3000 to your local machine.
Run below from the local Terminal:

```bash
gcloud compute ssh YOUR_INSTANCE \
    --zone YOUR_ZONE \
    --tunnel-through-iap \
    -- -L 3000:localhost:3000
```

Then open [http://localhost:3000](http://localhost:3000) in your browser. Keep the SSH session open while using the UI.

### What You Can Do in the Web UI

- **Chat interface** — Ask questions and watch the research pipeline execute in real time
- **Async jobs** — Submit long-running deep research tasks and retrieve results later
- **Knowledge upload** — Upload PDFs and documents to build a domain-specific knowledge base

The agent can now accessible through a browser and chat with users via multimodal inputs.

![AI-Q Architecture](./images/aiq_chat_ui.png)  

### 7. Clean up

Since we modified a file inside the AI-Q submodule, let's restore the original orchestrator prompt so the submodule stays clean:

In [ ]:
# Restore the original prompt (removes the tax domain header)
with open(prompt_path, "w") as f:
    f.write(original_prompt)

print("Original orchestrator prompt restored.")

> **Note**: The `prompt_path` and `original_prompt` variables were set earlier in this notebook. If you restarted the kernel, you can also restore the prompt by running `git checkout` inside the submodule:
> ```bash
> cd aiq && git checkout -- src/aiq_agent/agents/deep_researcher/prompts/orchestrator.j2
> ```

## Summary

In this notebook, we deployed Nemotron from Vertex AI Model Garden, built a domain-specific personal assistant with AI-Q, and deployed it as a web application using Docker Compose.

### Key Takeaways

- **Nemotron** is an open model family designed for agentic tasks
- **AI-Q** is a deep research agent blueprint built on the NeMo Agent Toolkit
- **Blueprints** are production-ready agent workflows you can run as-is or customize
- **Ran the agent** and observed its multi-stage process: intent classification → clarification → planning → research → cited report
- **Docker Compose deployment** packages the agent, proxy, frontend, and database into a production-ready stack accessible via Web UI